[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/badaouihakimou/machine-learning-notebooks/blob/main/05_dictionnaires.ipynb)



# Les dictionnaires

Une liste associe des positions à des valeurs : `villes[0]`, `villes[1]`.
Un dictionnaire associe des clés à des valeurs : `traduction['chien']`.

C'est la structure à utiliser dès que les données ont un nom plutôt qu'un rang.

## Le plan

| Section | Le sujet |
|---|---|
| 1 | Créer un dictionnaire, et ce qui peut servir de clé |
| 2 | Accéder aux valeurs, et éviter le `KeyError` |
| 3 | Ajouter, modifier, supprimer |
| 4 | `keys`, `values`, `items` |
| 5 | Parcourir un dictionnaire |
| 6 | `fromkeys` et son piège |
| 7 | Compter avec un dictionnaire |

## Deux pièges annoncés

Le premier vient d'une règle simple : une clé ne peut apparaître qu'une fois. Si
on l'écrit deux fois, la seconde écrase la première sans aucun message, et le
dictionnaire a moins d'entrées que prévu.

Le second concerne `fromkeys` avec une valeur mutable : toutes les clés
partagent alors le même objet.

Prérequis : les notebooks 02 à 04.

## 1. Créer un dictionnaire

Des accolades, et des paires `clé: valeur` séparées par des virgules.

In [75]:
traduction = {
    'chien': 'dog',
    'chat': 'cat',
    'souris': 'mouse',
    'oiseau': 'bird',
}

print(traduction)
print(len(traduction), 'entrées')

{'chien': 'dog', 'chat': 'cat', 'souris': 'mouse', 'oiseau': 'bird'}
4 entrées


La virgule après la dernière entrée est facultative, mais pratique : ajouter une
ligne ne demande pas de modifier la précédente.

Les valeurs peuvent être de n'importe quel type, y compris des listes ou d'autres
dictionnaires.

In [76]:
inventaire = {
    'bananes': 5000,
    'pommes': 456,
    'poires': 4354,
}

imbrique = {
    'dict_1': traduction,
    'dict_2': inventaire,
}

print(imbrique['dict_2'])
print(imbrique['dict_2']['pommes']) # deux niveaux

{'bananes': 5000, 'pommes': 456, 'poires': 4354}
456


### La règle fondamentale : une clé est unique

Une valeur peut se répéter autant qu'on veut, une clé non. Et c'est ici que se
cache le premier piège.

In [77]:
doublon = {
    'a': 1,
    'b': 2,
    'a': 99, # même clé que la première
}

print(doublon)
print(len(doublon), 'entrées au lieu de 3')

{'a': 99, 'b': 2}
2 entrées au lieu de 3


Aucune erreur, aucun avertissement. La seconde valeur écrase simplement la
première, et le dictionnaire compte une entrée de moins que le nombre de lignes
écrites.

Sur un petit dictionnaire ça se voit. Sur un gros, écrit sur vingt lignes, c'est
invisible.

In [78]:
import numpy as np
parametres = {
    'W1': np.random.randn(10, 100),
    'b1': np.random.randn(10, 1),
    'W2': np.random.randn(10, 10),
    'b1': np.random.randn(10, 1), # faute de frappe : devrait être b2
}

print('Clés  :', list(parametres.keys()))
print('Nombre:', len(parametres), 'au lieu de 4')

Clés  : ['W1', 'b1', 'W2']
Nombre: 3 au lieu de 4


Ce cas n'est pas théorique : c'est exactement la structure qu'on utilise pour
stocker les poids d'un réseau de neurones. La quatrième ligne aurait dû être
`b2`, et la faute de frappe supprime silencieusement `b1`.

Un réseau à qui il manque un paramètre ne plante pas forcément il apprend mal,
et on cherche l'erreur ailleurs pendant des heures.

La vérification qui coûte une ligne :

```python
assert len(parametres) == 4, 'Nombre de paramètres incorrect'
```

### Quelles valeurs peuvent servir de clé ?

Uniquement des objets immuables : chaînes, nombres, tuples, booléens.

In [79]:
valide = {
    'texte': 1,
    42: 'un nombre',
    (1, 2): 'un tuple',
    True: 'un booléen',
}
print(valide)

try:
    invalide = {[1, 2]: 'une liste'}
except TypeError as e:
    print('TypeError :', e)

{'texte': 1, 42: 'un nombre', (1, 2): 'un tuple', True: 'un booléen'}
TypeError : unhashable type: 'list'


Le message dit `unhashable type` : Python calcule une empreinte de la clé pour
la ranger, et cette empreinte doit rester stable. Une liste pouvant changer,
l'empreinte deviendrait fausse et la valeur serait introuvable.

C'est un argument concret en faveur des tuples, vus au notebook précédent : eux
peuvent servir de clé.

```python
distances = {('Paris', 'Lyon'): 465, ('Paris', 'Marseille'): 775}
```

## 2. Accéder aux valeurs

Avec des crochets, comme une liste, mais en donnant la clé.

In [80]:
print(traduction['chien'])
print(inventaire['pommes'])

dog
456


Si la clé n'existe pas, Python lève une erreur.

In [81]:
try:
    print(inventaire['peche'])
except KeyError as e:
    print('KeyError :', e)

KeyError : 'peche'


### get : la version qui ne plante pas

`get` renvoie `None` au lieu de lever une erreur.

In [82]:
print(inventaire.get('peche')) # None
print(inventaire.get('pommes')) # la valeur

None
456


Et on peut fournir une valeur de remplacement :

In [83]:
print(inventaire.get('peche', 0)) # 0 si absent
print(inventaire.get('pommes', 0)) # la vraie valeur si présent

0
456


C'est très utile pour un compteur ou un cumul :

```python
inventaire['peche'] = inventaire.get('peche', 0) + 10
```

Cette ligne fonctionne que la clé existe ou non.

Quand utiliser quoi. Les crochets si l'absence est une anomalie on veut que
ça plante pour le savoir. `get` si l'absence est un cas normal prévu.

### Tester la présence

In [84]:
print('pommes' in inventaire)
print('peche' in inventaire)
print('peche' not in inventaire)

True
False
True


`in` porte sur les clés, pas sur les valeurs. Pour chercher dans les valeurs,
il faut le préciser :

```python
print(5000 in inventaire.values())
```

In [85]:
print(5000 in inventaire.values())

True


## 3. Ajouter, modifier, supprimer

### Ajouter et modifier

C'est la même syntaxe, et c'est ce qui rend le piège des doublons possible.

In [86]:
inventaire = {'bananes': 5000, 'pommes': 456, 'poires': 4354}

inventaire['abricots'] = 1000 # clé absente : ajout
inventaire['pommes'] = 999 # clé présente : remplacement

print(inventaire)

{'bananes': 5000, 'pommes': 999, 'poires': 4354, 'abricots': 1000}


Rien ne distingue les deux opérations à l'écriture. Si tu veux ajouter sans
écraser :

```python
if 'pommes' not in inventaire:
    inventaire['pommes'] = 999
```

Ou avec `setdefault`, qui n'écrit que si la clé est absente :

In [87]:
inventaire.setdefault('pommes', 111) # existe déjà : ignoré
inventaire.setdefault('cerises', 222) # nouvelle clé : ajoutée

print(inventaire)

{'bananes': 5000, 'pommes': 999, 'poires': 4354, 'abricots': 1000, 'cerises': 222}


### update : fusionner deux dictionnaires

In [88]:
nouveaux = {'kiwis': 300, 'bananes': 6000}

inventaire.update(nouveaux)
print(inventaire)

{'bananes': 6000, 'pommes': 999, 'poires': 4354, 'abricots': 1000, 'cerises': 222, 'kiwis': 300}


`update` ajoute les clés absentes et écrase les clés existantes. Ici, `bananes`
passe de 5000 à 6000.

### Supprimer

In [89]:
inventaire = {'bananes': 5000, 'pommes': 456, 'poires': 4354, 'abricots': 1000}

retire = inventaire.pop('poires') # renvoie la valeur
print('pop a retiré :', retire)
print(inventaire)

del inventaire['abricots'] # ne renvoie rien
print(inventaire)

pop a retiré : 4354
{'bananes': 5000, 'pommes': 456, 'abricots': 1000}
{'bananes': 5000, 'pommes': 456}


`pop` sur une clé absente lève une `KeyError`, sauf si on donne une valeur par
défaut :

In [90]:
try:
    inventaire.pop('inexistant')
except KeyError as e:
    print('KeyError :', e)

print('avec défaut :', inventaire.pop('inexistant', 0))

KeyError : 'inexistant'
avec défaut : 0


Attention en notebook. `pop` modifie le dictionnaire. Réexécuter la cellule
une seconde fois retire une autre clé, ou lève une erreur.

C'est ce qui explique qu'un dictionnaire puisse perdre plus de clés que prévu :
la cellule a été lancée plusieurs fois. Redéfinis le dictionnaire au début de la
cellule si tu comptes la relancer.

`clear()` vide tout, avec la même mise en garde qu'au notebook précédent.

## 4. keys, values, items

Trois méthodes pour obtenir les composants d'un dictionnaire.

In [91]:
inventaire = {'bananes': 5000, 'pommes': 456, 'poires': 4354}

print(inventaire.keys())
print(inventaire.values())
print(inventaire.items())

dict_keys(['bananes', 'pommes', 'poires'])
dict_values([5000, 456, 4354])
dict_items([('bananes', 5000), ('pommes', 456), ('poires', 4354)])


Ce ne sont pas des listes mais des vues. Elles ne copient rien : elles
reflètent le dictionnaire en direct.

In [92]:
cles = inventaire.keys()
print('avant :', cles)

inventaire['kiwis'] = 300
print('après :', cles)  # la vue s'est mise à jour toute seule

avant : dict_keys(['bananes', 'pommes', 'poires'])
après : dict_keys(['bananes', 'pommes', 'poires', 'kiwis'])


C'est efficace, mais il y a une conséquence : on ne peut pas modifier un
dictionnaire pendant qu'on parcourt une de ses vues.

In [93]:
inventaire = {'a': 1, 'b': 2, 'c': 3}

try:
    for cle in inventaire.keys():
        if inventaire[cle] < 3:
            del inventaire[cle]
except RuntimeError as e:
    print('RuntimeError :', e)

RuntimeError : dictionary changed size during iteration


La solution est de parcourir une copie des clés :

In [94]:
inventaire = {'a': 1, 'b': 2, 'c': 3}

for cle in list(inventaire.keys()): # list() fige les clés
    if inventaire[cle] < 3:
        del inventaire[cle]

print(inventaire)

{'c': 3}


C'est le même problème que la suppression pendant un parcours de liste, vu au
notebook 03 sauf qu'ici Python lève une erreur au lieu de sauter des éléments
en silence. C'est un progrès.

Pour convertir une vue en liste : `list(inventaire.keys())`. Utile si on veut
trier, indexer, ou garder un état figé.

In [95]:
inventaire = {'bananes': 5000, 'pommes': 456, 'poires': 4354}

print(sorted(inventaire.keys()))
print(sum(inventaire.values()))
print(max(inventaire.values()))

['bananes', 'poires', 'pommes']
9810
5000


## 5. Parcourir un dictionnaire

Une boucle `for` directe parcourt les clés.

In [96]:
for cle in inventaire:
    print(cle)

bananes
pommes
poires


C'est équivalent à `for cle in inventaire.keys()`, en plus court.

Pour les valeurs :

In [97]:
for valeur in inventaire.values():
    print(valeur)

5000
456
4354


Et pour les deux à la fois, `items` renvoie des couples.

In [98]:
for paire in inventaire.items():
    print(paire, type(paire))

('bananes', 5000) <class 'tuple'>
('pommes', 456) <class 'tuple'>
('poires', 4354) <class 'tuple'>


Chaque élément est un tuple. On peut le déballer directement dans la boucle,
comme avec `enumerate` :

In [99]:
for fruit, quantite in inventaire.items():
    print(f'{fruit:<10} : {quantite:>6}')

bananes    :   5000
pommes     :    456
poires     :   4354


C'est la forme à retenir. Les noms `fruit` et `quantite` disent ce qu'ils
contiennent, contrairement à `k` et `v`.

Le `:<10` et `:>6` sont les formats d'alignement vus au notebook 02 pratique
pour aligner un tableau.

### L'ordre des clés

Depuis Python 3.7, un dictionnaire conserve l'ordre d'insertion. Les clés
ressortent dans l'ordre où elles ont été ajoutées, pas dans l'ordre
alphabétique.

C'est ce qui rend `dict.fromkeys()` utile pour dédoublonner une liste en gardant
l'ordre, comme vu à l'exercice 1 du notebook précédent.

Pour parcourir dans un autre ordre, il faut trier explicitement :

In [100]:
for fruit in sorted(inventaire):
    print(fruit, inventaire[fruit])

print()
# Trier par valeur décroissante
for fruit, qte in sorted(inventaire.items(), key=lambda paire: paire[1], reverse=True):
    print(f'{fruit:<10} {qte}')

bananes 5000
poires 4354
pommes 456

bananes    5000
poires     4354
pommes     456


Le `key=lambda paire: paire[1]` indique sur quoi trier : le deuxième élément de
chaque couple, donc la valeur. C'est l'usage typique des lambdas vu au notebook
02 une petite fonction passée en argument.

## 6. fromkeys et son piège

`fromkeys` construit un dictionnaire à partir d'une liste de clés, toutes
associées à la même valeur.

In [101]:
villes = ('Paris', 'Bruxelles', 'Londres')

print(dict.fromkeys(villes))
print(dict.fromkeys(villes, 'défaut'))
print(dict.fromkeys(villes, 0))

{'Paris': None, 'Bruxelles': None, 'Londres': None}
{'Paris': 'défaut', 'Bruxelles': 'défaut', 'Londres': 'défaut'}
{'Paris': 0, 'Bruxelles': 0, 'Londres': 0}


Note d'écriture : on l'appelle sur `dict`, pas sur un dictionnaire existant.
Écrire `inventaire.fromkeys(villes)` fonctionne, mais c'est trompeur ça ne
touche pas à `inventaire`, ça crée un dictionnaire indépendant. Autant écrire
`dict.fromkeys` pour que ce soit clair.

### Le piège

Avec une valeur mutable une liste, un dictionnaire toutes les clés
partagent le même objet.

In [102]:
compteurs = dict.fromkeys(('a', 'b', 'c'), [])

compteurs['a'].append(1)  # on ne touche qu'à 'a'

print(compteurs)
print("Même objet ?", compteurs['a'] is compteurs['b'])

{'a': [1], 'b': [1], 'c': [1]}
Même objet ? True


Les trois clés ont reçu le 1, alors qu'on n'en a modifié qu'une.

C'est encore la mutabilité du notebook 02 : `fromkeys` ne crée qu'une seule
liste et la range sous les trois clés. Elles pointent toutes vers le même objet
en mémoire.

Exactement le même mécanisme que le défaut mutable d'une fonction, où
`def f(x, liste=[])` partage une liste entre tous les appels.

La solution est une compréhension de dictionnaire, qui crée un objet neuf par
clé :

In [103]:
compteurs = {cle: [] for cle in ('a', 'b', 'c')}

compteurs['a'].append(1)

print(compteurs)
print("Même objet ?", compteurs['a'] is compteurs['b'])

{'a': [1], 'b': [], 'c': []}
Même objet ? False


Cette écriture `{cle: valeur for ...}` est une compréhension de dictionnaire,
le pendant de la compréhension de liste. Quelques exemples :

In [104]:
carres = {n: n**2 for n in range(1, 6)}
print(carres)

# Inverser un dictionnaire
inverse = {valeur: cle for cle, valeur in traduction.items()}
print(inverse)

# Filtrer
gros = {k: v for k, v in inventaire.items() if v > 1000}
print(gros)

{1: 1, 2: 4, 3: 9, 4: 16, 5: 25}
{'dog': 'chien', 'cat': 'chat', 'mouse': 'souris', 'bird': 'oiseau'}
{'bananes': 5000, 'poires': 4354}


Attention à l'inversion : si deux clés ont la même valeur, l'une des deux
disparaît c'est le piège de l'unicité des clés qui revient.

`fromkeys` reste parfait avec des valeurs immuables : `0`, `None`, `''`. Le
partage est alors sans danger puisqu'on ne peut pas les modifier.

## 7. Compter avec un dictionnaire

C'est l'usage le plus courant du dictionnaire en analyse de données : compter
les occurrences.

In [105]:
texte = 'bonjour tout le monde'

compteur = {}
for lettre in texte:
    compteur[lettre] = compteur.get(lettre, 0) + 1

print(compteur)

{'b': 1, 'o': 4, 'n': 2, 'j': 1, 'u': 2, 'r': 1, ' ': 3, 't': 2, 'l': 1, 'e': 2, 'm': 1, 'd': 1}


Le `get(lettre, 0)` est la clé de ce code. À la première rencontre d'une lettre,
la clé n'existe pas encore : `get` renvoie 0, on ajoute 1. Ensuite, il renvoie le
compte actuel.

Sans `get`, il faudrait un `if` :

```python
if lettre in compteur:
    compteur[lettre] += 1
else:
    compteur[lettre] = 1
```

### La version toute faite

Python propose un compteur dédié dans le module `collections`.

In [106]:
from collections import Counter

compteur = Counter(texte)
print(compteur)
print()
print('Les 3 plus fréquents :', compteur.most_common(3))
print("Occurrences de 'o' :", compteur['o'])
print('Lettre absente     :', compteur['z']) # 0, pas d'erreur

Counter({'o': 4, ' ': 3, 'n': 2, 'u': 2, 't': 2, 'e': 2, 'b': 1, 'j': 1, 'r': 1, 'l': 1, 'm': 1, 'd': 1})

Les 3 plus fréquents : [('o', 4), (' ', 3), ('n', 2)]
Occurrences de 'o' : 4
Lettre absente     : 0


`Counter` est un dictionnaire enrichi : il renvoie 0 pour une clé absente au lieu
de lever une erreur, et `most_common` trie par fréquence.

### defaultdict

Autre outil du même module, utile quand les valeurs sont des listes.

In [107]:
from collections import defaultdict

groupes = defaultdict(list) # crée une liste vide pour toute clé absente

animaux = [('mammifère', 'chien'), ('oiseau', 'moineau'),
           ('mammifère', 'chat'), ('oiseau', 'aigle')]

for categorie, nom in animaux:
    groupes[categorie].append(nom)

print(dict(groupes))

{'mammifère': ['chien', 'chat'], 'oiseau': ['moineau', 'aigle']}


Sans `defaultdict`, il faudrait vérifier l'existence de la clé avant chaque
`append`. Et c'est justement le cas où `fromkeys` avec une liste ne marche pas.

C'est exactement ce que fait `groupby` en Pandas, en plus élaboré : regrouper des
éléments par catégorie.

## 8. Mémo

### Liste ou dictionnaire

| | Liste | Dictionnaire |
|---|---|---|
| Accès par | position | clé |
| Ordre | oui | oui, depuis Python 3.7 |
| Doublons | autorisés | clés uniques |
| Recherche d'un élément | lente | immédiate |

Cette dernière ligne compte : chercher dans une liste de 100 000 éléments
demande de la parcourir, alors qu'un dictionnaire trouve la clé directement.

### Les méthodes

| Méthode | Effet |
|---|---|
| `d[cle]` | lit, lève `KeyError` si absent |
| `d.get(cle, defaut)` | lit sans erreur |
| `d[cle] = v` | ajoute ou remplace |
| `d.setdefault(cle, v)` | ajoute seulement si absent |
| `d.update(autre)` | fusionne |
| `d.pop(cle)` | retire et renvoie |
| `del d[cle]` | retire |
| `d.keys()`, `d.values()`, `d.items()` | les vues |

### Les pièges

| Situation | Ce qui se passe |
|---|---|
| Clé écrite deux fois | la seconde écrase, sans message |
| `fromkeys` avec une liste | toutes les clés partagent le même objet |
| Modifier pendant un parcours | `RuntimeError` |
| `pop` relancé en notebook | des clés disparaissent |
| Inverser un dict | les valeurs en double se perdent |
| Liste comme clé | `TypeError: unhashable` |

## 9. Exercices

**Exercice 1**

Écris une fonction `inverser(dictionnaire)` qui échange clés et valeurs. Teste-la
sur un dictionnaire contenant deux clés de même valeur, et explique ce qui se
passe. Propose ensuite une version qui ne perd rien, en regroupant les clés dans
une liste.

**Exercice 2**

À partir d'une phrase, écris une fonction `mots_frequents(texte, n)` qui renvoie
les n mots les plus fréquents, sans utiliser `Counter`. Ignore la casse et la
ponctuation. Compare ensuite ton résultat avec `Counter`.

**Exercice 3**

Cette fonction contient le piège de `fromkeys` :

```python
def creer_registres(noms):
    return dict.fromkeys(noms, [])
```

Montre le problème en ajoutant un élément à un seul registre. Explique pourquoi,
puis corrige-la. Vérifie avec `is`.

## Pour continuer

Le notebook suivant porte sur la compréhension de liste, entrevue ici avec les
compréhensions de dictionnaire.

Les dictionnaires reviendront constamment par la suite : ce sont eux qui portent
les hyperparamètres d'un modèle scikit-learn, les poids d'un réseau de neurones,
et ils se convertissent directement en DataFrame Pandas avec
`pd.DataFrame(mon_dict)`.

In [108]:
# Exercice 1
def inverser(dictionnaire):
    """Échange les clés et les valeurs d'un dictionnaire."""
    return {valeur: cle for cle, valeur in dictionnaire.items()}

In [109]:
traduction = {'chien': 'dog', 'chat': 'cat', 'souris': 'mouse'}
print(inverser(traduction))

{'dog': 'chien', 'cat': 'chat', 'mouse': 'souris'}


In [110]:
synonymes = {'chien': 'dog', 'chat': 'cat', 'matou': 'cat'}
print(inverser(synonymes))

{'dog': 'chien', 'cat': 'matou'}


In [111]:
print('avant :', len(synonymes), 'entrées')
print('après :', len(inverser(synonymes)), 'entrées')

avant : 3 entrées
après : 2 entrées


In [112]:
def inverser_complet(dictionnaire):
    """Inverse un dictionnaire en regroupant les clés de même valeur."""
    resultat = {}
    for cle, valeur in dictionnaire.items():
        resultat.setdefault(valeur, []).append(cle)
    return resultat

print(inverser_complet(synonymes))

{'dog': ['chien'], 'cat': ['chat', 'matou']}


In [113]:
from collections import defaultdict

def inverser_v3(dictionnaire):
    resultat = defaultdict(list)
    for cle, valeur in dictionnaire.items():
        resultat[valeur].append(cle)
    return dict(resultat)

In [114]:
print(inverser_v3(synonymes))

{'dog': ['chien'], 'cat': ['chat', 'matou']}


In [115]:
# Exercice 2

import string

def mots_frequents(texte, n=3):
    """Retourne les n mots les plus fréquents d'un texte.
    Ignore la casse et la ponctuation.
    Retourne une liste de couples (mot, occurrences), du plus au moins fréquent.
    """
    # Normalisation
    texte = texte.lower()
    for signe in string.punctuation:
        texte = texte.replace(signe, ' ')

    # Comptage
    compteur = {}
    for mot in texte.split():
        compteur[mot] = compteur.get(mot, 0) + 1

    # Tri décroissant
    tries = sorted(compteur.items(), key=lambda paire: paire[1], reverse=True)
    return tries[:n]

In [116]:
texte = "Le chat dort. Le chien dort aussi, et le chat ronronne !"

for mot, nombre in mots_frequents(texte, 4):
    print(f'{mot:<10} {nombre}')

le         3
chat       2
dort       2
chien      1


In [117]:
# Exercice 3

In [118]:
def creer_registres(noms):
    return dict.fromkeys(noms, [])

registres = creer_registres(['ventes', 'achats', 'stock'])

registres['ventes'].append('facture 001')

print(registres)

{'ventes': ['facture 001'], 'achats': ['facture 001'], 'stock': ['facture 001']}


In [119]:
print("ventes et achats, même objet ?", registres['ventes'] is registres['achats'])
print('id ventes :', id(registres['ventes']))
print('id achats :', id(registres['achats']))

ventes et achats, même objet ? True
id ventes : 139077403524416
id achats : 139077403524416


In [120]:
def creer_registres(noms):
    """Crée un registre indépendant pour chaque nom."""
    return {nom: [] for nom in noms}

In [121]:
registres = creer_registres(['ventes', 'achats', 'stock'])
registres['ventes'].append('facture 001')

print(registres)
print("même objet ?", registres['ventes'] is registres['achats'])

{'ventes': ['facture 001'], 'achats': [], 'stock': []}
même objet ? False
